# Importacao das libs

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession, Window
from pyspark import StorageLevel
import pyspark.sql.functions as F

from scipy.sparse import csr_matrix, save_npz, load_npz
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Criação de monitoramento com MlFLow

In [ ]:
import mlflow
from mlflow.models import infer_signature
# Define o banco de dados na raiz do projeto
mlflow.set_tracking_uri("sqlite:///../mlflow.db")

# Cria ou seleciona um experimento com nome específico
mlflow.set_experiment('tech-challenge-recomendacao-retailrocket')

In [ ]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("RetailRocket")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark.conf.set("spark.sql.shuffle.partitions", "48")

In [ ]:
df_events = spark.read.csv(
    "../data/interim/events.csv",
    header=True,
    inferSchema=True
)

df_prop_1 = spark.read.csv(
    "../data/interim/item_properties_part1.csv",
    header=True,
    inferSchema=True
)

df_prop_2 = spark.read.csv(
    "../data/interim/item_properties_part2.csv",
    header=True,
    inferSchema=True
)
df_prop = df_prop_1.unionByName(df_prop_2)

In [ ]:
# Criação do data frame de eventos e itens
window_spec = (
    Window
    .partitionBy("itemid", "property")
    .orderBy(F.col("timestamp").desc())
)

df_prop_last = (
    df_prop
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

df_items = (
    df_prop_last
    .filter(F.col("property").isin(["categoryid", "available"]))
    .groupBy("itemid")
    .pivot("property")
    .agg(F.first("value"))
)

df_eda = df_events.join(df_items, on="itemid", how="left")

## Funções de avaliação (compartilhadas por todos os modelos)

Centralizar as métricas aqui garante que Popularity, KNN, SVD e a MLP sejam avaliados
exatamente da mesma forma — mesma fórmula, mesmas chaves de dicionário, mesmo `k`.

In [ ]:
def precision_at_k(recomendados: list, relevantes: set, k: int) -> float:
    """Proporção de itens recomendados que são relevantes."""
    hits = len(set(recomendados[:k]) & relevantes)
    return hits / k


def recall_at_k(recomendados: list, relevantes: set, k: int) -> float:
    """Proporção de itens relevantes que foram recomendados."""
    if not relevantes:
        return 0.0
    hits = len(set(recomendados[:k]) & relevantes)
    return hits / len(relevantes)


def ndcg_at_k(recomendados: list, relevantes: set, k: int) -> float:
    """Normalized Discounted Cumulative Gain — penaliza hits tardios no ranking."""
    dcg = sum(
        1 / np.log2(rank + 2)
        for rank, item in enumerate(recomendados[:k])
        if item in relevantes
    )
    ideal = sum(1 / np.log2(i + 2) for i in range(min(len(relevantes), k)))
    return dcg / ideal if ideal > 0 else 0.0


def coverage(todas_recs: list[list], n_itens_total: int) -> float:
    """Proporção do catálogo que aparece em pelo menos uma recomendação."""
    unicos = {item for recs in todas_recs for item in recs}
    return len(unicos) / n_itens_total


def evaluate_model(
    recommend_fn,
    test_users: list,
    gt_dict: dict,
    n_items_total: int,
    k: int = 10,
) -> dict:
    """Avalia um modelo na MESMA população de usuários, com as MESMAS 4 métricas.

    recommend_fn: callable que recebe um visitorid (ID original, não índice)
                  e retorna uma list[int] de itemids recomendados.
    test_users:   lista de visitorid usados para avaliação — deve ser IDÊNTICA
                  entre todos os modelos para a comparação ser válida.
    gt_dict:      {visitorid: list de itemids relevantes (ground truth)}
    """
    precisions, recalls, ndcgs, all_recs = [], [], [], []

    for visitor_id in test_users:
        relevantes = set(gt_dict[visitor_id])
        if not relevantes:
            continue

        recs = recommend_fn(visitor_id)
        all_recs.append(recs)

        precisions.append(precision_at_k(recs, relevantes, k))
        recalls.append(recall_at_k(recs, relevantes, k))
        ndcgs.append(ndcg_at_k(recs, relevantes, k))

    return {
        "precision_at_10": float(np.mean(precisions)),
        "recall_at_10":    float(np.mean(recalls)),
        "ndcg_at_10":      float(np.mean(ndcgs)),
        "coverage":        coverage(all_recs, n_items_total),
        "n_usuarios_avaliados": len(precisions),
    }

## Preparação dos dados — interações ponderadas, filtro de cold users, split temporal

Esta célula monta a base que **todos os modelos vão compartilhar**: a mesma matriz
de interações, o mesmo conjunto de usuários ativos, o mesmo split treino/teste e
o mesmo `test_users` para avaliação.

In [ ]:
WEIGHTS_EXPR = (
    F.when(F.col("event") == "view", F.lit(1))
     .when(F.col("event") == "addtocart", F.lit(3))
     .when(F.col("event") == "transaction", F.lit(5))
     .otherwise(F.lit(0))
)

df_com_ts = (
    df_events
    .withColumn("peso", WEIGHTS_EXPR)
    .persist(StorageLevel.MEMORY_AND_DISK)
)

print(df_com_ts.count())
df_com_ts.printSchema()
df_com_ts.show(10)

In [ ]:
# Separação de treino e de teste
cutoff = df_com_ts.approxQuantile("timestamp", [0.8], 0.01)[0]

df_treino_raw = df_com_ts.filter(F.col("timestamp") <= cutoff)
df_teste_raw  = df_com_ts.filter(F.col("timestamp") >  cutoff)

print(f"Eventos treino (antes do filtro): {df_treino_raw.count():,}")
print(f"Eventos teste  (antes do filtro): {df_teste_raw.count():,}")


contagem_treino = (
    df_treino_raw
    .groupBy("visitorid")
    .agg(F.countDistinct("itemid").alias("n_itens"))
)

usuarios_ativos = (
    contagem_treino
    .filter(F.col("n_itens") >= 5)
    .select("visitorid")
    .cache()
)

n_usuarios_ativos = usuarios_ativos.count()
print(f"Usuários com >=5 interações NO TREINO: {n_usuarios_ativos:,}")


df_treino_spark = df_treino_raw.join(usuarios_ativos, "visitorid")
df_teste_spark  = df_teste_raw.join(usuarios_ativos, "visitorid")

interacoes_treino = (
    df_treino_spark
    .groupBy("visitorid", "itemid")
    .agg(F.sum("peso").alias("peso"))
    .persist(StorageLevel.MEMORY_AND_DISK)
)
interacoes_treino.count()

ground_truth = (
    df_teste_spark
    .groupBy("visitorid")
    .agg(F.collect_set("itemid").alias("itens_relevantes"))
)

gt_dict = {
    row["visitorid"]: row["itens_relevantes"]
    for row in ground_truth.collect()
}

print(f"Eventos treino (depois do filtro): {df_treino_spark.count():,}")
print(f"Eventos teste  (depois do filtro): {df_teste_spark.count():,}")
print(gt_dict)

In [ ]:
AMOSTRA_FRACAO = 0.60

usuarios_sample = (
    interacoes_treino
    .select("visitorid")
    .distinct()
    .sample(False, AMOSTRA_FRACAO, seed=SEED)
)

interacoes_sample = interacoes_treino.join(usuarios_sample, "visitorid")
ground_truth_sample = ground_truth.join(usuarios_sample, "visitorid")

print("Usuários na amostra:", usuarios_sample.count())
print("Interações na amostra:", interacoes_sample.count())

df_pd = interacoes_sample.toPandas()
gt_pd = ground_truth_sample.toPandas()

In [ ]:
# Mapeamentos ID -> índice contíguo, compartilhados por todos os modelos
unique_users = df_pd["visitorid"].unique()
unique_items = df_pd["itemid"].unique()

user_to_idx = {user: idx for idx, user in enumerate(unique_users)}
item_to_idx = {item: idx for idx, item in enumerate(unique_items)}
idx_to_user = {idx: user for user, idx in user_to_idx.items()}
idx_to_item = {idx: item for item, idx in item_to_idx.items()}

rows = df_pd["visitorid"].map(user_to_idx)
cols = df_pd["itemid"].map(item_to_idx)
data = df_pd["peso"].astype(np.float32)

matrix = csr_matrix(
    (data, (rows, cols)),
    shape=(len(unique_users), len(unique_items)),
)

print(f"Matriz: {matrix.shape}")

In [ ]:
# Reaproveita df_items do EDA (já tem categoryid via PySpark)
df_items_categoria = (
    df_items
    .select("itemid", "categoryid")
    .filter(F.col("categoryid").isNotNull())
)

df_categoria_pd = df_items_categoria.toPandas()
item_to_categoria_raw = dict(zip(df_categoria_pd["itemid"], df_categoria_pd["categoryid"]))

categorias_unicas = sorted(set(item_to_categoria_raw.values()) | {"desconhecida"})
categoria_to_idx = {cat: idx for idx, cat in enumerate(categorias_unicas)}

# Vetor de categoria por ÍNDICE de item, alinhado com idx_to_item da matriz
item_idx_to_categoria_idx = np.array([
    categoria_to_idx.get(
        item_to_categoria_raw.get(idx_to_item[i], "desconhecida"),
        categoria_to_idx["desconhecida"]
    )
    for i in range(len(idx_to_item))
])

n_categorias = len(categorias_unicas)
print(f"Categorias únicas: {n_categorias}")
print(f"Itens com categoria desconhecida: {(item_idx_to_categoria_idx == categoria_to_idx['desconhecida']).sum()}")

In [ ]:
def construir_ground_truth_valido(
    gt_dict: dict,
    user_to_idx: dict,
    matrix: csr_matrix,
    idx_to_item: dict
) -> dict:

    gt_valido = {}
    descartados_sem_item_novo = 0

    for visitor_id, itens_teste in gt_dict.items():

        user_idx = user_to_idx.get(visitor_id)

        if user_idx is None:
            continue

        itens_vistos_idx = matrix[user_idx].indices
        itens_vistos_ids = {
            idx_to_item[i]
            for i in itens_vistos_idx
        }

        itens_novos = list(
            set(itens_teste) - itens_vistos_ids
        )

        if itens_novos:
            gt_valido[visitor_id] = itens_novos
        else:
            descartados_sem_item_novo += 1

    print(f"Usuários no ground truth original: {len(gt_dict):,}")
    print(
        f"Usuários sem item novo no teste: "
        f"{descartados_sem_item_novo:,}"
    )
    print(
        f"Usuários válidos para avaliação: "
        f"{len(gt_valido):,}"
    )

    return gt_valido

In [ ]:
gt_valido = construir_ground_truth_valido(
    gt_dict,
    user_to_idx,
    matrix,
    idx_to_item
)

In [ ]:
N_SAMPLE_EVAL = 3_000
N_RECS = 10

gt_dict = dict(zip(gt_pd["visitorid"], gt_pd["itens_relevantes"]))

usuarios_candidatos = [u for u in gt_dict if u in user_to_idx]

rng = np.random.default_rng(SEED)
test_users = rng.choice(
    usuarios_candidatos,
    size=min(N_SAMPLE_EVAL, len(usuarios_candidatos)),
    replace=False,
).tolist()

print(f"Usuários de teste (compartilhado por todos os modelos): {len(test_users):,}")
print(test_users)

# Baseline 1 — Popularity

In [ ]:
# Score por item = soma dos pesos de TODAS as interações de treino
item_scores = np.asarray(matrix.sum(axis=0)).flatten()
top_items_global = [idx_to_item[i] for i in np.argsort(item_scores)[::-1][:N_RECS]]

print("Top 10 itens globais:", top_items_global)

In [35]:
def recomendar_popularity(visitor_id) -> list:
    """Recomendação idêntica para qualquer usuário — não personaliza."""
    return top_items_global


# --- Executa a avaliação do modelo popular ---
metricas_pop = evaluate_model(
    recommend_fn=recomendar_popularity,
    test_users=test_users,
    gt_dict=gt_dict,
    n_items_total=len(unique_items),
    k=N_RECS,
)

print("\n── Popularity Baseline ──────────────────")
for nome, valor in metricas_pop.items():
    print(f"  {nome:<24} {valor:.4f}")

# ==============================================================================
# MONITORAMENTO MLFLOW - BASELINE POPULARIDADE (CORRIGIDO)
# ==============================================================================
# Se df_eda for muito grande e estourar a memória, você pode passar apenas um subset (.head(100))
dataset = mlflow.data.from_pandas(gt_pd, name="dataset_RetailRocket_Events_and_Properties")

with mlflow.start_run(run_name="Baseline-Popularity") as run:

    mlflow.log_input(dataset, context="training/test")  

    # 1. Tags identificadoras do modelo
    mlflow.set_tags({
        "model_type": "popularity_baseline",
        "model_architecture": "Ranking de popularidade",
        "framework": "pandas_spark",
        "phase": "baseline",
        "dataset_name": "RetailRocket_Events_and_Properties",
        "dataset_size": f"{len(gt_pd)}",
        "feature_set": "Implicit_Feedback_with_Category_Order"
    })
    
    # 2. Parâmetros essenciais
    mlflow.log_params({
        "data.test_users_count": len(test_users),
        "recommendation.top_k": N_RECS,
        "catalog.total_items": len(unique_items)
    })
    
    # 3. Métricas extraídas do evaluate_model
    mlflow.log_metrics({
        "eval.precision_at_10": metricas_pop["precision_at_10"],
        "eval.recall_at_10": metricas_pop["recall_at_10"],
        "eval.ndcg_at_10": metricas_pop["ndcg_at_10"],
        "eval.catalog_coverage": metricas_pop["coverage"]
    })

    # 4. Assinatura do Modelo (Inputs e Outputs esperados no formato correto)
    # Input: Um ID de usuário / Output: Uma lista com os N_RECS itens recomendados
    sample_user = np.array([12345], dtype=np.int64)
    sample_output = np.array(top_items_global[:N_RECS], dtype=np.int64)
    
    signature = infer_signature(
        model_input={"visitor_id": sample_user},
        model_output=sample_output
    )
    
    # 5. Salvando a lista de itens mais populares do baseline
    # Como não é uma classe do Scikit-Learn, guardamos a lista como um dicionário JSON legível
    pop_dict = {"top_items_global": [int(x) for x in top_items_global]}
    mlflow.log_dict(pop_dict, artifact_file="ordenacao-popularidade/top_items.json")
    
    print("Corrida do MLflow finalizada e registrada com sucesso!")


── Popularity Baseline ──────────────────
  precision_at_10          0.0032
  recall_at_10             0.0065
  ndcg_at_10               0.0068
  coverage                 0.0002
  n_usuarios_avaliados     1814.0000


c:\Users\lara-\workspace\projeto-tech-challenge-sistema-recomendacao\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Corrida do MLflow finalizada e registrada com sucesso!


# Baseline 2 — KNN User-CF

In [ ]:
K_NEIGHBORS = 10

knn = NearestNeighbors(
    metric="cosine",
    algorithm="brute",   # único compatível com cosine + matriz esparsa
    n_neighbors=K_NEIGHBORS + 1,  # +1 porque o próprio usuário é seu vizinho mais próximo
    n_jobs=-1,
)
knn.fit(matrix)
print("KNN treinado.")

In [38]:
def recomendar_knn(visitor_id, n: int = N_RECS) -> list:
    """Recomendação por similaridade de cosseno entre usuários."""
    user_idx = user_to_idx[visitor_id]

    distances, neighbors = knn.kneighbors(matrix[user_idx])
    neighbor_idxs = neighbors[0][1:]          # remove o próprio usuário
    similarities  = 1 - distances[0][1:]       # cosseno: distância -> similaridade

    neighbor_matrix = matrix[neighbor_idxs]
    scores = np.asarray(similarities @ neighbor_matrix).ravel()

    ja_viu = matrix[user_idx].nonzero()[1]
    scores[ja_viu] = -np.inf

    top_idxs = np.argsort(scores)[::-1][:n]
    return [idx_to_item[i] for i in top_idxs]


# --- Executa a avaliação do KNN ---
metricas_knn = evaluate_model(
    recommend_fn=recomendar_knn,
    test_users=test_users,
    gt_dict=gt_dict,
    n_items_total=len(unique_items),
    k=N_RECS,
)

print("\n── KNN User-CF ──────────────────")
for nome, valor in metricas_knn.items():
    print(f"  {nome:<24} {valor:.4f}")

# ==============================================================================
# MONITORAMENTO MLFLOW - BASELINE KNN USER-CF (CORRIGIDO)
# ==============================================================================
dataset = mlflow.data.from_pandas(gt_pd, name="dataset_RetailRocket_Events_and_Properties")

with mlflow.start_run(run_name="Baseline-KNN-UserCF") as run:
    
    mlflow.log_input(dataset, context="training/test")      

    # 1. Tags identificadoras do modelo
    mlflow.set_tags({
        "model_type": "knn_collaborative_filtering",
        "model_architecture": "Knn_user_cf",
        "framework": "scikit-learn",
        "phase": "baseline",
        "dataset_name": "RetailRocket_Events_and_Properties",
        "dataset_size": f"{len(gt_pd)}",
        "feature_set": "Implicit_Feedback_Knn_user_cf"
    })

    # 2. Hiperparâmetros do KNN e metadados reais
    mlflow.log_params({
        "model.n_neighbors": 10,
        "model.metric": "cosine",
        "model.n_jobs": "-1",
        "recommendation.top_k": N_RECS,
        "data.test_users_count": len(test_users)
    })
    
    # 3. Métricas extraídas do evaluate_model
    mlflow.log_metrics({
        "eval.precision_at_10": metricas_knn["precision_at_10"],
        "eval.recall_at_10": metricas_knn["recall_at_10"],
        "eval.ndcg_at_10": metricas_knn["ndcg_at_10"],
        "eval.catalog_coverage": metricas_knn["coverage"]
    })

    # 4. Assinatura Corrigida (Inputs e Outputs condizentes com a realidade)
    sample_user = np.array([12345], dtype=np.int64)
    # Output simulado contendo uma lista de 10 IDs reais de itens do catálogo
    sample_output = np.array(list(idx_to_item.values())[:N_RECS], dtype=np.int64)
    
    signature = infer_signature(
        model_input={"visitor_id": sample_user},
        model_output=sample_output
    )
        
    # 5. Salvamento seguro das chaves de teste (Puxando diretamente do dicionário Python nativo)
    lista_usuarios_teste = [int(uid) for uid in gt_dict.keys()]
    pop_dict = {"test_users": lista_usuarios_teste}
    
    mlflow.log_dict(pop_dict, artifact_file="knn-user-cf/knn-user-cf.json")
    
    print("Corrida do MLflow finalizada e registrada com sucesso!")

c:\Users\lara-\workspace\projeto-tech-challenge-sistema-recomendacao\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\lara-\workspace\projeto-tech-challenge-sistema-recomendacao\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\lara-\workspace\projeto-tech-challenge-sistema-recomendacao\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the curre


── KNN User-CF ──────────────────
  precision_at_10          0.0055
  recall_at_10             0.0130
  ndcg_at_10               0.0103
  coverage                 0.1594
  n_usuarios_avaliados     1814.0000


c:\Users\lara-\workspace\projeto-tech-challenge-sistema-recomendacao\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Corrida do MLflow finalizada e registrada com sucesso!


# Baseline 3 — SVD (matrix factorization)

In [39]:
N_COMPONENTS = 50

svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=SEED)
user_factors = svd.fit_transform(matrix)   # (n_users, n_components)
item_factors = svd.components_.T           # (n_items, n_components)

print(f"Variância explicada acumulada: {svd.explained_variance_ratio_.sum():.2f}")

Variância explicada acumulada: 0.32


In [40]:
def recomendar_svd(visitor_id, n: int = N_RECS) -> list:
    """Recomendação pelo produto interno entre fatores latentes de usuário e item."""
    user_idx = user_to_idx[visitor_id]

    scores = user_factors[user_idx] @ item_factors.T
    itens_vistos = matrix[user_idx].indices
    scores[itens_vistos] = -np.inf

    # Garante eficiência O(N) na busca dos maiores elementos antes da ordenação final
    top_idxs = np.argpartition(scores, -n)[-n:]
    top_idxs = top_idxs[np.argsort(scores[top_idxs])[::-1]]

    return [idx_to_item[i] for i in top_idxs]


# --- Executa a avaliação do SVD ---
metricas_svd = evaluate_model(
    recommend_fn=recomendar_svd,
    test_users=test_users,
    gt_dict=gt_dict,
    n_items_total=len(unique_items),
    k=N_RECS,
)

print("\n── SVD ──────────────────")
for nome, valor in metricas_svd.items():
    print(f"  {nome:<24} {valor:.4f}")

# ==============================================================================
# MONITORAMENTO MLFLOW - BASELINE SVD (CORRIGIDO)
# ==============================================================================
dataset = mlflow.data.from_pandas(gt_pd, name="dataset_RetailRocket_Events_and_Properties")

with mlflow.start_run(run_name="Baseline-SVD") as run:
    mlflow.log_input(dataset, context="training/test")      
    
    # 1. Tags identificadoras do modelo
    mlflow.set_tags({
        "model_type": "svd_matrix_factorization",
        "model_architecture": f"TruncatedSVD_with_{N_COMPONENTS}_components",
        "framework": "scikit-learn",
        "phase": "baseline",
        "dataset_name": "RetailRocket_Events_and_Properties",
        "dataset_size": f"{len(gt_pd)}",
        "feature_set": "Implicit_Feedback_SVD_Matrix_Factorization"
    })
    
    # 2. Hiperparâmetros do SVD
    mlflow.log_params({
        "model.n_components": N_COMPONENTS,
        "model.random_state": SEED,
        "recommendation.top_k": N_RECS,
        "data.test_users_count": len(test_users)
    })
    
    # 3. Métricas extraídas do evaluate_model
    mlflow.log_metrics({
        "eval.precision_at_10": metricas_svd["precision_at_10"],
        "eval.recall_at_10": metricas_svd["recall_at_10"],
        "eval.ndcg_at_10": metricas_svd["ndcg_at_10"],
        "eval.catalog_coverage": metricas_svd["coverage"]
    })

    # 4. Assinatura Corrigida (Inputs com o nome correto do parâmetro e Output com array de IDs)
    sample_user = np.array([12345], dtype=np.int64)
    sample_output = np.array(list(idx_to_item.values())[:N_RECS], dtype=np.int64)
    
    signature = infer_signature(
        model_input={"visitor_id": sample_user},
        model_output=sample_output
    )
        
    # 5. Salvamento seguro do dicionário de IDs utilizando chaves limpas em Python
    lista_usuarios_teste = [int(uid) for uid in gt_dict.keys()]
    pop_dict = {"test_users": lista_usuarios_teste}
    
    mlflow.log_dict(pop_dict, artifact_file="svd/svd.json")
    
    print("Corrida do MLflow finalizada e registrada com sucesso!")


── SVD ──────────────────
  precision_at_10          0.0017
  recall_at_10             0.0029
  ndcg_at_10               0.0028
  coverage                 0.0116
  n_usuarios_avaliados     1814.0000
Corrida do MLflow finalizada e registrada com sucesso!


c:\Users\lara-\workspace\projeto-tech-challenge-sistema-recomendacao\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


In [41]:
comparacao = pd.DataFrame([
    {"Modelo": "Popularidade", **metricas_pop},
    {"Modelo": "KNN User-CF",  **metricas_knn},
    {"Modelo": "SVD",          **metricas_svd},
]).round(4)

comparacao = comparacao.sort_values("ndcg_at_10", ascending=False).reset_index(drop=True)
comparacao

,Modelo,precision_at_10,recall_at_10,ndcg_at_10,coverage,n_usuarios_avaliados
0,KNN User-CF,0.0055,0.0130,0.0103,0.1594,1814
1,Popularidade,0.0032,0.0065,0.0068,0.0002,1814
2,SVD,0.0017,0.0029,0.0028,0.0116,1814
